# ME324 · Lab 10 — Text-gen 3/4: attention & transformers

**Lecture 10 · "Building your own text-generating AI (3/4)" · 2026-08-17**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsrobinson/me324/blob/main/labs/lab-10-transformer-gpt.ipynb)

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

---

> **Turn the GPU on first.** In Colab: **Runtime → Change runtime type → Hardware
> accelerator → GPU**. Strongly recommended today — you will train a real (tiny)
> transformer.

### Today you will build a GPT.

Same data, same training loop, same interface as Labs 8 and 9 — but today the model
becomes a **transformer**, the architecture behind every modern LLM (GPT, Claude,
Llama, ...). You build it from a single attention head up, mirroring Karpathy's
`nanoGPT`, then race it against the Lab-8 bigram and the Lab-9 RNN on the *same* data.

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** the four lines of attention by hand, then `Head`, `Block`, and the full `GPTLanguageModel`, then train it and sample.
- **Stretch / take-home — skip if short on time:** the parameter hand-count, the three-way bigram/RNN/GPT comparison, and the attention heatmap.

_Most of the code is written for you; the `# TODO` cells are the parts you write. Worked answers are in the **Solutions** section at the bottom._

## Run me first — setup

Imports, a device pick (GPU if Colab gave you one), and the course seed — matching
Labs 8 and 9.

In [ ]:
import os, urllib.request
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

# --- device handling: use the GPU if Colab gave us one ---
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():   # Apple Silicon, if running locally
    device = "mps"
else:
    device = "cpu"
print("Using device:", device)

# Reproducibility: the same seed every run (matches Labs 8 and 9).
torch.manual_seed(1337)


## Section 1 · The Lab-8 pipeline, reused

Every model in Labs 8–11 has used the same interface, so the pipeline, training loop
and sampler can carry over unchanged:

```python
logits, loss = model(idx, targets=None)    # idx, targets: (B, T) longs;  logits: (B, T, vocab)
idx = model.generate(idx, max_new_tokens)  # (B, T) -> (B, T + max_new_tokens)
```

`B` = batch size, `T` = context length, `vocab` = number of distinct characters.
First, let's get the corpus — the complete works of Shakespeare, exactly as in Lab 8.

In [ ]:
url = ("https://raw.githubusercontent.com/karpathy/char-rnn/"
       "master/data/tinyshakespeare/input.txt")
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("length of dataset in characters:", len(text))
print("---- first 200 characters ----")
print(text[:200])


### The knobs

A transformer earns a longer context and a wider embedding than the bigram, so
`block_size` rises to 32 and `n_embd` to 64. These four dials (`block_size`, `n_embd`,
`n_head`, `n_layer`) set a transformer's capacity — scaled up ~10,000×, they give you
GPT-4.

In [ ]:
block_size = 32    # context length: how many characters the model sees at once
batch_size = 32    # how many sequences we train on in parallel
n_embd     = 64    # embedding width (the "d" in the lecture)
n_head     = 4     # number of attention heads per block
n_layer    = 4     # number of transformer blocks (the depth of the stack)
dropout    = 0.0   # regularisation; 0.0 keeps this short run simple

head_size  = n_embd // n_head   # each head works in this many dimensions
print(f"block_size={block_size}, n_embd={n_embd}, n_head={n_head}, "
      f"n_layer={n_layer}, head_size={head_size}")


### The pipeline function (from Lab 8)

Here's Lab 8's pipeline, collected into one function. It returns `encode` / `decode`
(text ↔ token ids), `get_batch('train')` (a random `(B, T)` batch), and
`estimate_loss(model)` (average train/val loss).

In [ ]:
def build_char_pipeline(text, block_size=8, batch_size=32, device="cpu", seed=1337):
    """Char-level pipeline: tokenizer, train/val split, batcher, loss estimator."""
    torch.manual_seed(seed)
    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: "".join(itos[i] for i in l)

    data = torch.tensor(encode(text), dtype=torch.long)
    n = int(0.9 * len(data))
    train_data, val_data = data[:n], data[n:]

    def get_batch(split):
        d = train_data if split == "train" else val_data
        ix = torch.randint(len(d) - block_size, (batch_size,))
        x = torch.stack([d[i:i + block_size] for i in ix])
        y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
        return x.to(device), y.to(device)

    @torch.no_grad()
    def estimate_loss(model, eval_iters=200):
        out = {}
        model.eval()
        for split in ["train", "val"]:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                xb, yb = get_batch(split)
                _, loss = model(xb, yb)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    return dict(vocab_size=vocab_size, stoi=stoi, itos=itos, encode=encode,
                decode=decode, get_batch=get_batch, estimate_loss=estimate_loss,
                block_size=block_size, device=device)


Build it and check the shapes: `x` and `y` are both `(batch_size, block_size)`, and
`y[t]` is the character that follows `x[t]`.

In [ ]:
P = build_char_pipeline(text, block_size=block_size, batch_size=batch_size, device=device)
vocab_size = P["vocab_size"]
encode, decode = P["encode"], P["decode"]
get_batch, estimate_loss = P["get_batch"], P["estimate_loss"]

print("vocab size:", vocab_size)
print("encode('hi there') ->", encode("hi there"))
print("decode back        ->", decode(encode("hi there")))

xb, yb = get_batch("train")
print("inputs  x shape:", tuple(xb.shape), "  (batch_size, block_size)")
print("targets y shape:", tuple(yb.shape))


### The training loop (from Lab 8)

The generic AdamW loop reused by every model in Labs 8–11 — we don't need to change anything here either!

In [ ]:
def train_model(model, get_batch, estimate_loss, max_iters=3000, eval_interval=500,
                lr=1e-3, device="cpu"):
    """Generic AdamW training loop, reused by every model in Labs 8-11."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss(model)
            print(f"step {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return model


## Section 2 · Build attention, one step at a time

* **2a** — one self-attention `Head`
* **2b** — `MultiHeadAttention`
* **2c** — `FeedForward`
* **2d** — `Block` (attention + feed-forward, with residuals + LayerNorm)
* **2e** — `GPTLanguageModel` (embeddings → blocks → output head)

### 2a · One attention head — the heart of the transformer

Recall the lecture: three learned matrices project each token's embedding into a
**query**, a **key** and a **value** (the code comments below gloss their roles).
Attention is a **soft lookup**: queries are scored against keys, and the output is a
weighted average of the values.

**The four lines of attention** (this is the whole operation):

1. **Scores** — every query dotted with every key: $QK^{\mathsf T}/\sqrt{d_k}$, a
   $(T\times T)$ matrix. The $\sqrt{d_k}$ stops dot products blowing up in high dimensions.
2. **Causal mask** — position $t$ must **not** see the future: set every entry where
   $j > i$ to $-\infty$ *before* the softmax, so those weights become 0.
3. **Softmax** — turn each row of scores into weights that sum to 1.
4. **Weighted sum of values** — $\mathbf{A}\mathbf{V}$.

Your turn: steps 1–3 on a tiny example. Step 2 is a good one to look up using an LLM — try
*"In PyTorch, how do I set the upper triangle of a matrix to -inf?"*

In [ ]:
torch.manual_seed(1337)

# A toy "sequence": batch of 1, T=4 tokens, each a C=2-dim embedding.
B, T, C = 1, 4, 2
hs = 2                      # head size
x = torch.randn(B, T, C)

# Learned projections: turn each token into a query, a key, and a value.
key   = nn.Linear(C, hs, bias=False)
query = nn.Linear(C, hs, bias=False)
value = nn.Linear(C, hs, bias=False)

k = key(x)      # (B, T, hs)  "what do I offer?"
q = query(x)    # (B, T, hs)  "what am I looking for?"
v = value(x)    # (B, T, hs)  "what will I contribute?"

# The four lines of attention -- steps 1-3 are yours, step 4 is given.
scores = ____    # TODO 1: every query dotted with every key -> (B, T, T), scaled by
                 #         1/sqrt(hs). (k.transpose(-2, -1) swaps the last two dims.)

tril = torch.tril(torch.ones(T, T))       # 1s on and below the diagonal, 0s above
scores = ____    # TODO 2: mask the future -- fill every position where tril == 0
                 #         with float("-inf"). Look up `masked_fill`.

weights = ____   # TODO 3: softmax the scores into weights -- along which dimension?

out = weights @ v                         # step 4 (given): weighted sum of the values

print("attention weights  (row i = how much token i attends to each token):")
print(weights[0])
print()
print("row sums  :", weights[0].sum(dim=-1))
print("out shape :", tuple(out.shape))

# CHECK: lower-triangular (no peeking ahead) and each row sums to 1.
assert torch.allclose(weights[0].sum(dim=-1), torch.ones(T)), "rows must sum to 1"
assert (weights[0].triu(diagonal=1) == 0).all(), "the future (upper triangle) must be 0"
print("\nPASSED: weights are lower-triangular and each row sums to 1.")

**What you should see.** Lower-triangular weights (no token peeks at the future), each
row summing to 1. Row 0 can only attend to token 0, so it is `[1, 0, 0, 0]`. That *is*
causal self-attention. One more check that you believe your own softmax:

In [ ]:
# TODO: before you run this cell, predict what the second line will print.
# Then run it, and write down why in the comment at the bottom.
wrong = F.softmax(scores, dim=-2)      # softmax down the COLUMNS instead of along the rows
print("row sums with dim=-1 :", weights[0].sum(dim=-1))
print("row sums with dim=-2 :", wrong[0].sum(dim=-1))

# Your answer -- what does row i of the weight matrix mean, and why does that
# force the softmax to run along dim=-1?
#

### Package it as a module

Same four lines, wrapped in an `nn.Module` so we can reuse and stack them: the
projections become `nn.Linear` layers, the mask a fixed `tril` buffer. The spec is in
the docstring.

In [ ]:
class Head(nn.Module):
    """One head of causal self-attention.

    TODO -- implement __init__ and forward:

    __init__(self, head_size):
        * self.key, self.query, self.value -- three nn.Linear(n_embd, head_size, bias=False)
        * register_buffer("tril", ...) -- lower-triangular ones, shape (block_size, block_size)
        * self.dropout = nn.Dropout(dropout)

    forward(self, x):    # x is (B, T, n_embd)
        The four lines you wrote in the worked example, with two adjustments:
        * slice the mask to the sequence you were actually given: self.tril[:T, :T]
        * apply self.dropout to the weights after the softmax
        Returns wei @ v, shape (B, T, head_size).
    """

    def __init__(self, head_size):
        super().__init__()
        raise NotImplementedError("Implement Head.__init__")

    def forward(self, x):
        raise NotImplementedError("Implement Head.forward")

In [ ]:
# CHECK: a head maps (B, T, n_embd) -> (B, T, head_size) and is shape-preserving in T.
torch.manual_seed(1337)
h = Head(head_size)
xb_demo = torch.randn(4, block_size, n_embd)     # (B, T, n_embd)
out = h(xb_demo)
print("Head output shape:", tuple(out.shape), "(expected (4, %d, %d))" % (block_size, head_size))
assert out.shape == (4, block_size, head_size)
print("PASSED.")


### 2b · Multi-head attention

One head learns one notion of relevance; we run `n_head` of them in **parallel** so
they can specialise (one tracks subject–verb agreement, another who "he" refers to,
...), then **concatenate** their outputs and **project** back to `n_embd`. Heads of
size `n_embd // n_head` concatenate back to width `n_embd` — shape-preserving, so it
stacks.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Several heads of self-attention in parallel, concatenated and projected."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = ____    # TODO 1: run every head on x and concatenate the results. Along
                      #         which dimension do 4 outputs of (B, T, 16) become (B, T, 64)?
        return ____   # TODO 2: project back to n_embd, then apply dropout.

In [ ]:
# CHECK: multi-head attention is shape-preserving: (B, T, n_embd) -> (B, T, n_embd).
torch.manual_seed(1337)
mha = MultiHeadAttention(n_head, head_size)
out = mha(xb_demo)
print("MultiHeadAttention output shape:", tuple(out.shape))
assert out.shape == (4, block_size, n_embd)
print("PASSED.")


### 2c · Feed-forward (the per-position MLP)

Attention mixes information *across* positions; the feed-forward layer then processes
each position *independently* — expand 4×, ReLU, project back — giving the model a lot
of capacity.

In [ ]:
class FeedForward(nn.Module):
    """A simple per-position MLP: expand 4x, ReLU, project back."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
# CHECK: feed-forward is shape-preserving too.
torch.manual_seed(1337)
ff = FeedForward(n_embd)
out = ff(xb_demo)
print("FeedForward output shape:", tuple(out.shape))
assert out.shape == (4, block_size, n_embd)
print("PASSED.")


### 2d · The transformer block

A block stitches attention and feed-forward together with the two ideas that make deep
stacks trainable:

* **Residual connections** — *add* each sub-layer's output to its input,
  $\mathbf{x} \leftarrow \mathbf{x} + f(\mathbf{x})$; the identity path keeps
  gradients flowing.
* **Pre-LayerNorm** — normalise each token's vector *before* each sub-layer; it trains
  more stably than post-norm.

```
x = x + attention(LayerNorm(x))     # communicate across positions
x = x + feedforward(LayerNorm(x))   # process each position
```

In [ ]:
class Block(nn.Module):
    """A transformer block: attention + feed-forward, each wrapped in a
    residual connection and a pre-LayerNorm.

    TODO -- implement __init__ and forward:

    __init__(self, n_embd, n_head):
        * self.sa   = MultiHeadAttention(n_head, n_embd // n_head)
        * self.ffwd = FeedForward(n_embd)
        * self.ln1, self.ln2 -- two separate nn.LayerNorm(n_embd)

    forward(self, x):
        Two residual updates -- the two-line sketch in the markdown above is the
        spec. Norm BEFORE each sub-layer, and don't lose the `x +`. Return x.
    """

    def __init__(self, n_embd, n_head):
        super().__init__()
        raise NotImplementedError("Implement Block.__init__")

    def forward(self, x):
        raise NotImplementedError("Implement Block.forward")

In [ ]:
# CHECK: a block is shape-preserving, so we can stack n_layer of them.
torch.manual_seed(1337)
blk = Block(n_embd, n_head)
out = blk(xb_demo)
print("Block output shape:", tuple(out.shape))
assert out.shape == (4, block_size, n_embd)
print("PASSED.")


### 2e · The full GPT

Token embeddings look up *what* each character is; positional embeddings are **added**
to say *where* it sits — without them, attention could not tell "dog bites man" from
"man bites dog". Then the stack of blocks, a final LayerNorm, and a linear head
producing next-token **logits**. `generate` (given) crops the context to the last
`block_size` tokens — the positional table has no more rows than that.

In [ ]:
class GPTLanguageModel(nn.Module):
    """The full GPT.

    TODO -- implement __init__ and forward (generate is given):

    __init__(self, vocab_size):
        * self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        * self.position_embedding_table = nn.Embedding(block_size, n_embd)
        * self.blocks  = nn.Sequential of n_layer Block(n_embd, n_head)
        * self.ln_f    = nn.LayerNorm(n_embd)
        * self.lm_head = nn.Linear(n_embd, vocab_size)

    forward(self, idx, targets=None):    # idx is (B, T) token ids
        * look up token embeddings, then ADD position embeddings
          (torch.arange(T, device=idx.device) gives the positions)
        * pass through self.blocks, then self.ln_f, then self.lm_head -> logits (B, T, vocab)
        * loss: None if targets is None; otherwise F.cross_entropy on the flattened
          logits and targets -- the same reshape as the bigram in Lab 8
        * return logits, loss
    """

    def __init__(self, vocab_size):
        super().__init__()
        raise NotImplementedError("Implement GPTLanguageModel.__init__")

    def forward(self, idx, targets=None):
        raise NotImplementedError("Implement GPTLanguageModel.forward")

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        # GIVEN: autoregressive sampling. Crops context to the last block_size tokens.
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
# CHECK: forward returns (logits, loss) with the right shapes; generate runs.
torch.manual_seed(1337)
_tmp = GPTLanguageModel(vocab_size).to(device)
xb, yb = get_batch("train")
logits, loss = _tmp(xb, yb)
print("logits shape:", tuple(logits.shape), " (expected (batch, block, vocab))")
print("loss:", round(loss.item(), 4), " (random-init loss ~ ln(vocab) =", round(__import__("math").log(vocab_size), 4), ")")
assert logits.shape == (batch_size, block_size, vocab_size)

ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
sample = decode(_tmp.generate(ctx, max_new_tokens=20)[0].tolist())
print("untrained sample (gibberish, as expected):", repr(sample))
print("parameter count:", sum(p.numel() for p in _tmp.parameters()))
del _tmp
print("PASSED.")


## Section 3 · Train, sample, and compare

Time to train — a couple of minutes on a Colab GPU.

In [ ]:
MAX_ITERS = 3000        # training steps. ~2-3 min on a GPU. On CPU, try 1000.
LEARNING_RATE = 1e-3

torch.manual_seed(1337)
gpt = GPTLanguageModel(vocab_size).to(device)
gpt_params = sum(p.numel() for p in gpt.parameters())
print(f"GPT parameter count: {gpt_params:,}\n")

train_model(gpt, get_batch, estimate_loss,
            max_iters=MAX_ITERS, eval_interval=500, lr=LEARNING_RATE, device=device)


### Generate some Shakespeare

Seed with a single newline (token id 0) and generate 500 characters — the same two
lines as Labs 8 and 9. Expect real **words**, **line breaks** and **capitalised
names**, but not actual Shakespeare.

In [ ]:
# Sampling, exactly as in Labs 8 and 9.
context = ____   # TODO 1: a (1, 1) tensor of zeros (token id 0 = '\n'), dtype long, on `device`

# TODO 2: ask `gpt` for 500 new tokens, decode them, and print the result.


### Where do those parameters live?

The training cell printed the parameter count — now check it by hand, the same habit
as Lab 5's 81-parameter net. There are three blanks, but we've provided the per-block 
arithmetic for you.

In [ ]:
# Fill in the blanks using the knobs: vocab_size, n_embd, block_size, head_size,
# n_head, n_layer.

tok_emb = ____   # TODO 1: the token-embedding table -- one n_embd-wide row per character
pos_emb = ____   # TODO 2: the position-embedding table -- one row per context position

# One block, worked for you -- read each term against the classes you wrote:
per_head  = 3 * (n_embd * head_size)                        # W_Q, W_K, W_V  (bias=False)
attn      = n_head * per_head + (n_embd * n_embd + n_embd)  # heads + output projection
ffwd      = (n_embd * 4*n_embd + 4*n_embd) + (4*n_embd * n_embd + n_embd)
norms     = 2 * (2 * n_embd)                                # ln1, ln2: a weight and a bias per feature
one_block = attn + ffwd + norms

final_ln = 2 * n_embd
lm_head  = ____   # TODO 3: the output Linear(n_embd, vocab_size) -- don't forget its bias

expected = tok_emb + pos_emb + n_layer * one_block + final_ln + lm_head
print(f"hand count: {expected:,}   PyTorch: {gpt_params:,}")
assert expected == gpt_params, "a term is missing or double-counted"
print("PASSED: every parameter accounted for.")

### The three-way comparison — same data, three models

Now retrain the Lab-8 **bigram** and Lab-9 **RNN** on the identical
pipeline and compare all three. Notice that lower val loss = better 
next-character prediction.

In [ ]:
# The two earlier models, repeated here so we can train them on the SAME data.

class BigramLanguageModel(nn.Module):
    """Lab 8: predicts the next char from the current char only."""
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)             # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


class RNNLanguageModel(nn.Module):
    """Lab 9: a GRU carries a hidden state across the sequence."""
    def __init__(self, vocab_size, n_embd=64, n_hidden=128):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.rnn = nn.GRU(n_embd, n_hidden, batch_first=True)
        self.lm_head = nn.Linear(n_hidden, vocab_size)

    def forward(self, idx, targets=None):
        emb = self.token_embedding_table(idx)                # (B, T, n_embd)
        out, _ = self.rnn(emb)                               # (B, T, n_hidden)
        logits = self.lm_head(out)                           # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [ ]:
def train_fresh(make_model, name):
    torch.manual_seed(1337)
    model = make_model().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"--- training {name}  ({n_params:,} params) ---")
    train_model(model, get_batch, estimate_loss,
                max_iters=MAX_ITERS, eval_interval=500, lr=LEARNING_RATE, device=device)
    return model, n_params

# Retrain the two earlier models on the same pipeline (GPT is already trained).
bigram, bigram_params = train_fresh(lambda: BigramLanguageModel(vocab_size), "Bigram (Lab 8)")
print()
rnn, rnn_params = train_fresh(lambda: RNNLanguageModel(vocab_size), "RNN (Lab 9)")


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
trio = [("BIGRAM (Lab 8)", bigram, bigram_params),
        ("RNN    (Lab 9)", rnn,    rnn_params),
        ("GPT    (Lab 10)", gpt,   gpt_params)]

for name, model, n_params in trio:
    final = estimate_loss(model)
    sample = decode(model.generate(context, 300)[0].tolist())
    print("=" * 72)
    print(f"{name}   |   {n_params:,} params   |   val loss {final['val']:.3f}")
    print("-" * 72)
    print(sample)
    print()


## Section 4 (optional) · Visualise an attention head

Plot block 0, head 0's weight matrix for a short phrase: row $i$ = how much token $i$
attends to each earlier token $j$. Note the lower-triangular mask, and look for
columns that "light up" — tokens many positions attend to.

In [ ]:
@torch.no_grad()
def head0_attention(model, idx):
    """Recompute the attention weights of block 0, head 0 for an input idx (1, T)."""
    B, T = idx.shape
    tok = model.token_embedding_table(idx)
    pos = model.position_embedding_table(torch.arange(T, device=idx.device))
    x = tok + pos
    block0 = model.blocks[0]
    xn = block0.ln1(x)                 # the head sees the pre-norm input
    head = block0.sa.heads[0]
    k, q = head.key(xn), head.query(xn)
    wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
    wei = wei.masked_fill(head.tril[:T, :T] == 0, float("-inf"))
    return F.softmax(wei, dim=-1)[0]   # (T, T)

phrase = "To be, or not to"
idx = torch.tensor([encode(phrase)], dtype=torch.long, device=device)
W = head0_attention(gpt, idx).cpu()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(W, cmap="viridis")
ax.set_xticks(range(len(phrase))); ax.set_xticklabels(list(phrase))
ax.set_yticks(range(len(phrase))); ax.set_yticklabels(list(phrase))
ax.set_xlabel("attends to (key position j)")
ax.set_ylabel("query position i")
ax.set_title("Block 0, Head 0 attention weights")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()


## Recap

**You built a GPT**: a self-attention head — $\text{softmax}(QK^{\mathsf T}/\sqrt{d_k})\,V$
with a causal mask — run in parallel, wrapped into pre-norm residual blocks, assembled
into **`GPTLanguageModel`**, and trained past the bigram and the RNN on the same data.
This is the architecture behind every modern LLM; what changes at the frontier is
mostly *scale*, not the design (although look up MoE models if you have the time!)

### Extensions (optional)

* Double `n_layer` or `n_embd`. How much does val loss improve — and training time?
* Set `dropout = 0.1` and train for 5000 iters. Does the train/val gap change?
* Raise `block_size` to 64 and re-run from the knobs cell. Attention cost grows as
  $T^2$ — can you feel it?

### Next time — Lab 11

A **BPE tokenizer** (the sub-word tokenizer real LLMs use), **sampling controls**
(`temperature`, `top-k`, `top-p` — your `generate` already has the hooks), then the
capstone: Karpathy's pure-Python micro-GPT on the very `Value` class you wrote in
Labs 3–4 — proving today's GPT is "just" your `Value` class plus loops.

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — the four lines of attention (steps 1–3)**

`hs ** -0.5` is $1/\sqrt{d_k}$; a chatbot may offer `/ math.sqrt(hs)` — identical.
`masked_fill` wants a boolean, hence `tril == 0`.

In [ ]:
torch.manual_seed(1337)

# A toy "sequence": batch of 1, T=4 tokens, each a C=2-dim embedding.
B, T, C = 1, 4, 2
hs = 2                      # head size
x = torch.randn(B, T, C)

# Learned projections: turn each token into a query, a key, and a value.
key   = nn.Linear(C, hs, bias=False)
query = nn.Linear(C, hs, bias=False)
value = nn.Linear(C, hs, bias=False)

k = key(x)      # (B, T, hs)  "what do I offer?"
q = query(x)    # (B, T, hs)  "what am I looking for?"
v = value(x)    # (B, T, hs)  "what will I contribute?"

# (1) SCORES: every query dotted with every key, scaled by sqrt(d_k).
scores = q @ k.transpose(-2, -1) * hs ** -0.5            # (B, T, T)

# (2) CAUSAL MASK: hide the future. Keep the lower triangle; set the rest to -inf.
tril = torch.tril(torch.ones(T, T))
scores = scores.masked_fill(tril == 0, float("-inf"))    # (B, T, T)

# (3) SOFTMAX over the last dimension -> weights that sum to 1 along each row.
weights = F.softmax(scores, dim=-1)                      # (B, T, T)

# (4) WEIGHTED SUM of the values.
out = weights @ v                                        # (B, T, hs)

print("attention weights  (row i = how much token i attends to each token):")
print(weights[0])
print()
print("row sums  :", weights[0].sum(dim=-1))
print("out shape :", tuple(out.shape))

# CHECK: lower-triangular (no peeking ahead) and each row sums to 1.
assert torch.allclose(weights[0].sum(dim=-1), torch.ones(T)), "rows must sum to 1"
assert (weights[0].triu(diagonal=1) == 0).all(), "the future (upper triangle) must be 0"
print("\nPASSED: weights are lower-triangular and each row sums to 1.")

**Solution — which dimension for the softmax? (predict, then run)**

Row $i$ is token $i$'s probability distribution over the positions it may attend to,
so the normalisation must run *along each row*: `dim=-1`. `dim=-2` normalises each
*column* instead (a key's popularity across queries) — the rows then no longer sum
to 1, and row 0, which should be exactly `[1, 0, 0, 0]`, isn't even a distribution.

**Solution — `Head`**

The slice `self.tril[:T, :T]` matters: `generate` starts from sequences shorter than
`block_size`, and the mask has to shrink with them.

In [ ]:
class Head(nn.Module):
    """One head of causal self-attention -- the worked example as a module."""

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # a fixed lower-triangular matrix used for the causal mask
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                            # (B, T, head_size)
        q = self.query(x)                                          # (B, T, head_size)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5        # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)                                          # (B, T, head_size)
        return wei @ v                                             # (B, T, head_size)


**Solution — `MultiHeadAttention.forward`**

`dim=-1` concatenates along the feature dimension: four `(B, T, 16)` head outputs
become one `(B, T, 64)` tensor, which `proj` then mixes back together.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Several heads of self-attention in parallel, concatenated and projected."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # concat over the head dim
        return self.dropout(self.proj(out))                   # project back to n_embd


**Solution — `Block`**

The `x +` is the whole point: drop it and the model still runs, but you have deleted
the residual path, and a deep stack trains far worse.

In [ ]:
class Block(nn.Module):
    """A transformer block: attention + feed-forward, each with a
    residual connection and a pre-LayerNorm."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))      # residual around attention   (pre-norm)
        x = x + self.ffwd(self.ln2(x))    # residual around feed-forward (pre-norm)
        return x


**Solution — `GPTLanguageModel`**

Note `device=idx.device` inside the `torch.arange` — forget it and the position
indices stay on the CPU while everything else is on the GPU, and the forward pass
falls over.

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                              # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = tok_emb + pos_emb                                                  # (B, T, n_embd)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                               # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]            # crop context to the last block_size tokens
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature    # focus on the last step
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


**Solution — sampling from the trained model**

The same two lines as Labs 8 and 9 — the shared interface at work.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(gpt.generate(context, max_new_tokens=500)[0].tolist()))


**Solution — the parameter count**

The blanks are the two embedding tables and the output head. Notice where the budget
goes: the four blocks hold ~95% of the 209,729 parameters.

In [ ]:
tok_emb = vocab_size * n_embd            # 65 * 64 = 4,160
pos_emb = block_size * n_embd            # 32 * 64 = 2,048

per_head  = 3 * (n_embd * head_size)                        # W_Q, W_K, W_V  (bias=False)
attn      = n_head * per_head + (n_embd * n_embd + n_embd)  # heads + output projection
ffwd      = (n_embd * 4*n_embd + 4*n_embd) + (4*n_embd * n_embd + n_embd)
norms     = 2 * (2 * n_embd)                                # ln1, ln2: a weight and a bias per feature
one_block = attn + ffwd + norms                             # 49,792

final_ln = 2 * n_embd
lm_head  = n_embd * vocab_size + vocab_size                 # weights + biases = 4,225

expected = tok_emb + pos_emb + n_layer * one_block + final_ln + lm_head
print(f"hand count: {expected:,}   PyTorch: {gpt_params:,}")
assert expected == gpt_params
print("PASSED: every parameter accounted for.")